# Fetching Historical Energy Usage Data

Reverse engineering Yale's energy usage website

In [1]:
import json
import re
import time
from io import BytesIO

import polars as pl
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

In [2]:
facilities = pl.read_csv("../data/raw/facilities.csv")
facids = facilities["facid"].to_list()

In [6]:
def is_numeric_string_col(series: pl.Series) -> bool:
    """Check if a String column actually contains numeric data."""
    non_null = series.drop_nulls()
    if len(non_null) == 0:
        return False
    sample = non_null.head(50)
    cast_result = sample.cast(pl.Float64, strict=False)
    non_null_after = cast_result.drop_nulls().len()
    return non_null_after / len(sample) > 0.8


response = requests.get("https://java.facilities.yale.edu/energy/")
soup = BeautifulSoup(response.text, "html.parser")
script_text = soup.find("script", string=re.compile("var user")).string
user_json = re.search(r'var user\s*=\s*(\{.*?\});', script_text, re.S).group(1)
user = json.loads(user_json)

cookies = {
    "JSESSIONID": user["jsessionID"],
    "token": user["token"],
}

dfs = []
for facid in tqdm(facids):
    url = (
        f'https://java.facilities.yale.edu/energy/queryresultsdownload?facids=["{facid}"]&startDate="1900-01-01T10:00:00.000Z"&endDate="2026-03-01T11:00:00.000Z"&style=series&commodityCode=all'
    )

    try:
        time.sleep(1)
        r = requests.get(url, cookies=cookies)
        r.raise_for_status()

        if not r.headers.get("content-type", "").startswith("application/vnd"):
            print(f"{facid}: request returned HTML or error, skipping")
            continue

        df = pl.read_excel(BytesIO(r.content))

        # Cast string columns to Float64 only if they look numeric
        df = df.with_columns([
            pl.col(c).cast(pl.Float64, strict=False)
            for c in df.columns
            if df[c].dtype == pl.String and is_numeric_string_col(df[c])
        ])

        # Cast any integer columns to Float64 for consistency
        df = df.with_columns([
            pl.col(c).cast(pl.Float64)
            for c in df.columns
            if df[c].dtype in (pl.Int64, pl.Int32, pl.Int16, pl.Int8)
        ])

        # Add facid as first column
        df = df.with_columns(
            pl.lit(facid).alias("facid")
        ).select(["facid", *df.columns])

        if "Usage Month" in df.columns and df.height > 0:
            dfs.append(df)

    except Exception as e:
        print(f"{facid}: error — {e}")
        continue

if dfs:
    energy_usage = pl.concat(dfs, how="vertical_relaxed")
    energy_usage.write_csv("../data/raw/energy_usage.csv")
    print(f"Saved {len(dfs)} facilities to ../data/raw/energy_usage.csv")
else:
    print("No data downloaded.")

100%|██████████| 370/370 [11:47<00:00,  1.91s/it]

Saved 295 facilities to ../data/raw/energy_usage.csv
